# 06 — Neural networks

Three further neural-network detectors, all trained from scratch on the canonical
split and registered in the serving stack (`ensembleWeight: 0.5` each):

- **MLP** (`pytorch-mlp`) — feed-forward network over the chi2-reduced TF-IDF + style
  vector (the feature layout the classical models consume), selector travels in the checkpoint.
- **CharCNN** (`pytorch-charcnn`) — the TextCNN architecture over a *character*
  vocabulary: convolutional n-gram detectors over raw characters, no tokenizer.
- **TinyTransformer** (`pytorch-tinytransformer`) — a small Transformer encoder from
  scratch (2 layers, 4 heads) over the shared word vocabulary, masked mean-pooling.


In [ ]:
import sys
from pathlib import Path

# Make `aitextd` importable regardless of the notebook working directory.
ROOT_HINT = Path.cwd()
while ROOT_HINT != ROOT_HINT.parent and not (ROOT_HINT / 'aitextd' / 'paths.py').exists():
    ROOT_HINT = ROOT_HINT.parent
sys.path.insert(0, str(ROOT_HINT))

from aitextd.paths import PROJECT_ROOT

FORCE_RETRAIN = False  # True: retrain all three models and rewrite their models.json entries

DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_JSON = PROJECT_ROOT / 'models.json'

RANDOM_SEED = 42
DEVICE = 'cpu'


In [ ]:
import joblib
import numpy as np
import pandas as pd
import torch

torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(12)

df = pd.read_csv(DATA_DIR / 'processedDataset.csv')
texts = df['text'].astype(str).to_numpy()
tokenized = joblib.load(DATA_DIR / 'tokenized_texts.joblib')
X = joblib.load(DATA_DIR / 'tfidf_vectorizer_X.joblib')
split = joblib.load(DATA_DIR / 'split_indices.joblib')
train_idx, test_idx = split['train_idx'], split['test_idx']
y = df['aiGenerated'].to_numpy()
y_train, y_test = y[train_idx], y[test_idx]
print(f'{len(df)} documents | train {len(train_idx)} / test {len(test_idx)}')


## Evaluation helper


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score


def expected_calibration_error(y_true, probs, bins=15):
    y_true, probs = np.asarray(y_true, dtype=float), np.asarray(probs, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probs > lo) & (probs <= hi)
        if mask.any():
            ece += mask.sum() / len(probs) * abs(y_true[mask].mean() - probs[mask].mean())
    return round(float(ece), 4)


def evaluate(y_true, probs):
    probs = np.asarray(probs, dtype=float)
    pred = (probs >= 0.5).astype(int)
    confusion = {
        'tn': int(((pred == 0) & (y_true == 0)).sum()), 'fp': int(((pred == 1) & (y_true == 0)).sum()),
        'fn': int(((pred == 0) & (y_true == 1)).sum()), 'tp': int(((pred == 1) & (y_true == 1)).sum()),
    }
    return {
        'accuracy': round(float(accuracy_score(y_true, pred)), 4),
        'precision': round(float(precision_score(y_true, pred)), 4),
        'recall': round(float(recall_score(y_true, pred)), 4),
        'f1': round(float(f1_score(y_true, pred)), 4),
        'roc_auc': round(float(roc_auc_score(y_true, probs)), 4),
        'ece': expected_calibration_error(y_true, probs),
        'confusion': confusion,
    }


def predict_probs(model, X, batch=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(torch.sigmoid(model(X[i : i + batch])).numpy())
    return np.concatenate(out)


def fit_loop(model, X_train_t, y_train_t, X_eval, y_eval, epochs, batch, lr):
    """BCEWithLogitsLoss + Adam; per-epoch monitoring, best-F1 weights kept."""
    import torch.nn as nn

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    best_f1, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(X_train_t))
        total = 0.0
        for start in range(0, len(perm), batch):
            idx = perm[start : start + batch]
            optimizer.zero_grad()
            loss = criterion(model(X_train_t[idx]), y_train_t[idx])
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        f1 = f1_score(y_eval, (predict_probs(model, X_eval) >= 0.5).astype(int))
        print(f'epoch {epoch}/{epochs}  loss={total / len(perm):.4f}  test_f1={f1:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    return model


## MLP over chi2-reduced TF-IDF + style

The classical models' joint feature layout, reduced to 10,000 chi2-selected TF-IDF
columns (+ the 10 style columns, always passed through), feeding a two-hidden-layer
perceptron. The fitted selector is saved inside the checkpoint so the serving path
rebuilds the exact feature pipeline.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2

from aitextd import paths
from aitextd.architectures import MLPClassifier

CHI2_K, HIDDEN_DIMS, MLP_DROPOUT = 10_000, (512, 128), 0.3
MLP_FILE = MODELS_DIR / 'mlpModel.pt'

if MLP_FILE.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(MLP_FILE, map_location=DEVICE, weights_only=False)
    selector, mlp_config = ckpt['selector'], ckpt['config']
    mlp = MLPClassifier(input_dim=mlp_config['input_dim'], hidden_dims=tuple(mlp_config['hidden_dims']), dropout=mlp_config['dropout'])
    mlp.load_state_dict(ckpt['state_dict'])
    mlp.eval()
    X_test_mlp = torch.tensor(selector.transform(X[test_idx]).toarray(), dtype=torch.float32)
    print('loaded checkpoint mlpModel.pt')
else:
    tfidf_dim = len(joblib.load(paths.TFIDF_VECTORIZER_JOBLIB).get_feature_names_out())
    selector = ColumnTransformer(
        [('tfidf_select', SelectKBest(chi2, k=CHI2_K), slice(0, tfidf_dim)),
         ('style', 'passthrough', slice(tfidf_dim, None))],
        sparse_threshold=1.0,
    )
    selector.fit(X[train_idx], y_train)
    X_train_mlp = torch.tensor(selector.transform(X[train_idx]).toarray(), dtype=torch.float32)
    X_test_mlp = torch.tensor(selector.transform(X[test_idx]).toarray(), dtype=torch.float32)
    mlp_config = {'input_dim': int(X_train_mlp.shape[1]), 'hidden_dims': list(HIDDEN_DIMS),
                  'dropout': MLP_DROPOUT, 'style_feature_count': int(X.shape[1] - tfidf_dim), 'chi2_k': CHI2_K}
    mlp = MLPClassifier(input_dim=mlp_config['input_dim'], hidden_dims=HIDDEN_DIMS, dropout=MLP_DROPOUT)
    mlp = fit_loop(mlp, X_train_mlp, torch.tensor(y_train, dtype=torch.float32), X_test_mlp, y_test, epochs=6, batch=128, lr=1e-3)
    torch.save({'state_dict': mlp.state_dict(), 'selector': selector, 'config': mlp_config}, MLP_FILE)
    print('saved', MLP_FILE.name)

mlp_metrics = evaluate(y_test, predict_probs(mlp, X_test_mlp))
print('MLP v1 test:', mlp_metrics)


## CharCNN

The TextCNN architecture over a character vocabulary built from the raw train texts
(no tokenizer, no lemmatization): convolution kernels see character n-grams directly.
Sequences are clamped to 512 characters.


In [ ]:
from collections import Counter

from aitextd.architectures import TextCNNClassifier

CHAR_MAX_LEN, PAD_IDX, UNK_IDX = 512, 0, 1
CHARCNN_FILE = MODELS_DIR / 'charcnnModel.pt'


def build_char_vocab(idx):
    counter = Counter()
    for i in idx:
        counter.update(texts[i])
    vocab = {'<pad>': PAD_IDX, '<unk>': UNK_IDX}
    for ch, freq in counter.most_common(1000):
        if freq < 2:
            break
        if ch not in vocab:
            vocab[ch] = len(vocab)
    return vocab


def encode_chars(text, vocab, max_len=CHAR_MAX_LEN):
    ids = [vocab.get(ch, UNK_IDX) for ch in text[:max_len]]
    return ids + [PAD_IDX] * (max_len - len(ids))


def encode_chars_split(idx, vocab):
    return torch.tensor([encode_chars(texts[i], vocab) for i in idx], dtype=torch.long)


def make_charcnn(vocab_size, config):
    return TextCNNClassifier(
        vocab_size, config['embed_dim'], config['n_filters'], tuple(config['kernel_sizes']),
        pad_idx=config['pad_idx'], dropout=config['dropout'],
    )


charcnn_config = {'embed_dim': 64, 'n_filters': 128, 'kernel_sizes': [3, 4, 5],
                  'dropout': 0.3, 'pad_idx': PAD_IDX, 'unk_idx': UNK_IDX, 'max_len': CHAR_MAX_LEN}

if CHARCNN_FILE.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(CHARCNN_FILE, map_location=DEVICE, weights_only=False)
    char_vocab, charcnn_config = ckpt['vocab'], ckpt['config']
    charcnn = make_charcnn(len(char_vocab), charcnn_config)
    charcnn.load_state_dict(ckpt['state_dict'])
    charcnn.eval()
    X_test_char = encode_chars_split(test_idx, char_vocab)
    print('loaded checkpoint charcnnModel.pt')
else:
    char_vocab = build_char_vocab(train_idx)
    X_train_char, X_test_char = encode_chars_split(train_idx, char_vocab), encode_chars_split(test_idx, char_vocab)
    charcnn = make_charcnn(len(char_vocab), charcnn_config)
    charcnn = fit_loop(charcnn, X_train_char, torch.tensor(y_train, dtype=torch.float32), X_test_char, y_test, epochs=4, batch=128, lr=1e-3)
    torch.save({'state_dict': charcnn.state_dict(), 'vocab': char_vocab, 'config': charcnn_config}, CHARCNN_FILE)
    print('saved', CHARCNN_FILE.name)

charcnn_metrics = evaluate(y_test, predict_probs(charcnn, X_test_char))
print('CharCNN v1 test:', charcnn_metrics)


## TinyTransformer (from scratch)

A 2-layer Transformer encoder (4 heads, feed-forward 256, learned positional
embeddings) over the same word vocabulary the other vocabulary-based models use —
no pretraining. Real tokens mask both the attention (padding) and the mean-pool.


In [ ]:
from aitextd.architectures import TinyTransformerClassifier

WORD_MAX_LEN, WORD_PAD, WORD_UNK = 200, 0, 1
TT_FILE = MODELS_DIR / 'tinyTransformerModel.pt'


def build_word_vocab(idx):
    counter = Counter()
    for i in idx:
        counter.update(tokenized[i])
    vocab = {'<pad>': WORD_PAD, '<unk>': WORD_UNK}
    for word, freq in counter.most_common(30_000):
        if freq < 2:
            break
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab


def encode_words(tokens, vocab, max_len=WORD_MAX_LEN):
    ids = [vocab.get(t, WORD_UNK) for t in tokens[:max_len]]
    return ids + [WORD_PAD] * (max_len - len(ids))


def encode_words_split(idx, vocab):
    return torch.tensor([encode_words(tokenized[i], vocab) for i in idx], dtype=torch.long)


def make_tinytransformer(vocab_size, config):
    return TinyTransformerClassifier(
        vocab_size, config['embed_dim'], config['n_layers'], config['n_heads'],
        config['ff_dim'], config['max_len'], pad_idx=config['pad_idx'], dropout=config['dropout'],
    )


tt_config = {'embed_dim': 128, 'n_layers': 2, 'n_heads': 4, 'ff_dim': 256,
             'max_len': WORD_MAX_LEN, 'pad_idx': WORD_PAD, 'unk_idx': WORD_UNK, 'dropout': 0.3}

if TT_FILE.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(TT_FILE, map_location=DEVICE, weights_only=False)
    tt_vocab, tt_config = ckpt['vocab'], ckpt['config']
    tinytransformer = make_tinytransformer(len(tt_vocab), tt_config)
    tinytransformer.load_state_dict(ckpt['state_dict'])
    tinytransformer.eval()
    X_test_tt = encode_words_split(test_idx, tt_vocab)
    print('loaded checkpoint tinyTransformerModel.pt')
else:
    tt_vocab = build_word_vocab(train_idx)
    X_train_tt, X_test_tt = encode_words_split(train_idx, tt_vocab), encode_words_split(test_idx, tt_vocab)
    tinytransformer = make_tinytransformer(len(tt_vocab), tt_config)
    tinytransformer = fit_loop(tinytransformer, X_train_tt, torch.tensor(y_train, dtype=torch.float32), X_test_tt, y_test, epochs=3, batch=32, lr=1e-3)
    torch.save({'state_dict': tinytransformer.state_dict(), 'vocab': tt_vocab, 'config': tt_config}, TT_FILE)
    print('saved', TT_FILE.name)

tinytransformer_metrics = evaluate(y_test, predict_probs(tinytransformer, X_test_tt))
print('TinyTransformer v1 test:', tinytransformer_metrics)


## Results on the canonical test split


In [ ]:
v1_results = {'mlp': mlp_metrics, 'charcnn': charcnn_metrics, 'tinytransformer': tinytransformer_metrics}
for model_id, m in v1_results.items():
    c = m['confusion']
    print(f"{model_id:16s} acc={m['accuracy']:.4f} f1={m['f1']:.4f} roc_auc={m['roc_auc']:.4f} "
          f"ece={m['ece']:.4f} confusion tn={c['tn']} fp={c['fp']} fn={c['fn']} tp={c['tp']}")


## Sanity check on the v2 corpus

The same cross-domain probe the other notebooks use: 1,200 balanced documents from the
`cross_domain__news` test split of the leak-free v2 corpus. CharCNN reads raw
characters; TinyTransformer reads lemmatized tokens; the MLP needs the joint feature
vector built on the fly (TF-IDF + style features computed per document).


In [ ]:
from aitextd.features import compute_style_features, style_features_vector
from aitextd.text import tokenize_persian_text

SUBSAMPLE_PER_CLASS = 600

df_v2 = pd.read_parquet(DATA_DIR / 'dataset_v2.parquet').reset_index(drop=True)
splits_v2 = joblib.load(DATA_DIR / 'splits_v2.joblib')
v2_idx = np.asarray(splits_v2['cross_domain__news__test'])
labels_v2 = df_v2['label_binary'].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
neg_pool, pos_pool = v2_idx[labels_v2[v2_idx] == 0], v2_idx[labels_v2[v2_idx] == 1]
keep = np.concatenate([
    rng.choice(neg_pool, size=min(SUBSAMPLE_PER_CLASS, len(neg_pool)), replace=False),
    rng.choice(pos_pool, size=min(SUBSAMPLE_PER_CLASS, len(pos_pool)), replace=False),
])
docs_v2 = df_v2.iloc[keep]
y_v2 = docs_v2['label_binary'].to_numpy()
texts_v2 = docs_v2['text'].astype(str).tolist()
tokens_v2 = [tokenize_persian_text(t) for t in texts_v2]
print(f'v2 cross_domain__news sanity set: {len(keep)} docs '
      f'(human={int((y_v2 == 0).sum())}, ai={int((y_v2 == 1).sum())})')

v2_results = {}
X_char_v2 = torch.tensor([encode_chars(t, char_vocab) for t in texts_v2], dtype=torch.long)
v2_results['charcnn'] = evaluate(y_v2, predict_probs(charcnn, X_char_v2))

X_tokens_v2 = torch.tensor([encode_words(t, tt_vocab) for t in tokens_v2], dtype=torch.long)
v2_results['tinytransformer'] = evaluate(y_v2, predict_probs(tinytransformer, X_tokens_v2))

from scipy import sparse as sp

style_rows = np.vstack([style_features_vector(compute_style_features(t, t, joblib.load(paths.TFIDF_VECTORIZER_JOBLIB))) for t in texts_v2])
X_joint_v2 = sp.hstack([joblib.load(paths.TFIDF_VECTORIZER_JOBLIB).transform(texts_v2), style_rows], format='csr')
X_mlp_v2 = torch.tensor(selector.transform(X_joint_v2).toarray(), dtype=torch.float32)
v2_results['mlp'] = evaluate(y_v2, predict_probs(mlp, X_mlp_v2))

for model_id, m in v2_results.items():
    print(f"{model_id:16s} acc={m['accuracy']:.4f} f1={m['f1']:.4f} roc_auc={m['roc_auc']:.4f}")


## Registration (only with FORCE_RETRAIN)

Writes the three `models.json` entries in the serving format (runtime strings match
`aitextd/loaders.py`); hand-tuned ensemble weights survive a retrain.


In [ ]:
def metrics_json(m):
    return {'accuracy': m['accuracy'], 'precision': m['precision'], 'recall': m['recall'],
            'f1Score': m['f1'], 'rocAuc': m['roc_auc']}


def confusion_json(m):
    c = m['confusion']
    return {'trueNegative': c['tn'], 'falsePositive': c['fp'], 'falseNegative': c['fn'], 'truePositive': c['tp']}


def build_entry(model_id, name, name_fa, description, description_fa, algorithm,
                filename, runtime, metrics, confusion, feature_count):
    return {
        'id': model_id, 'name': name, 'nameFa': name_fa,
        'description': description, 'descriptionFa': description_fa,
        'algorithm': algorithm, 'type': 'dl', 'version': '1.0',
        'filename': filename, 'runtime': runtime, 'ensembleWeight': 0.5,
        'metrics': metrics, 'confusionMatrix': confusion,
        'training': {
            'totalSamples': len(train_idx) + len(test_idx), 'trainSamples': len(train_idx),
            'testSamples': len(test_idx), 'featureCount': feature_count,
            'testSplitRatio': 0.2, 'randomSeed': RANDOM_SEED,
            'dataset': 'persian_ai_generated_final.csv + blogs.csv',
        },
    }


ENTRIES = [
    build_entry(
        'mlp', 'MLP', 'پرسپترون چندلایه',
        'A two-hidden-layer perceptron over the chi2-reduced TF-IDF and stylometric feature '
        'vector (the same joint layout the classical models consume, reduced to 10,000 TF-IDF '
        'columns). The fitted feature selector travels inside the checkpoint so the serving '
        'path rebuilds the exact train-time feature pipeline. Trained from scratch on CPU on '
        'the canonical split to distinguish AI-generated content from human writing.',
        'پرسپترون چندلایه با دو لایهٔ پنهان روی بردار TF-IDF کاهش‌یافته با آزمون chi2 و ویژگی‌های سبک‌شناسی '
        '(همان چیدمان مشترک مدل‌های کلاسیک که به ۱۰ هزار ستون TF-IDF کاهش می‌یابد). انتخاب‌گر ویژگی آموخته‌شده '
        'درون چک‌پوینت ذخیره می‌شود تا مسیر سرویس، خط‌لولهٔ ویژگی زمان آموزش را عیناً بازسازی کند. از ابتدا روی '
        'CPU و تفکیک مرجع آموزش دیده تا متن تولیدشده توسط هوش مصنوعی را از نوشته انسان جدا کند.',
        'MLP (2 hidden layers)', 'mlpModel.pt', 'pytorch-mlp',
        metrics_json(mlp_metrics), confusion_json(mlp_metrics), mlp_config['input_dim'],
    ),
    build_entry(
        'charcnn', 'CharCNN', 'شبکه کانولوشنی نویسه‌ای',
        'The TextCNN architecture applied at the character level: a character vocabulary built '
        'from the raw train texts (no tokenizer, no lemmatization) feeds parallel 1D convolutions '
        'with kernel sizes 3, 4 and 5 over 512-character windows, followed by ReLU, a length-masked '
        'global max-pooling per kernel, dropout and a linear head. Trained from scratch on CPU on '
        'the canonical split to distinguish AI-generated content from human writing.',
        'معماری TextCNN در سطح نویسه: واژگان نویسه‌ای ساخته‌شده از متن خام مجموعهٔ آموزش (بدون توکنیزه‌کننده و '
        'بدون ریشه‌یابی) به کانولوشن‌های یک‌بعدی موازی با اندازه‌های ۳ و ۴ و ۵ روی پنجره‌های ۵۱۲ نویسه‌ای می‌رود و '
        'پس از آن‌ها ReLU، بیشینه‌گیری سراسری با ماسک طول برای هر هسته، dropout و سر خطی قرار دارد. از ابتدا روی '
        'CPU و تفکیک مرجع آموزش دیده تا متن تولیدشده توسط هوش مصنوعی را از نوشته انسان جدا کند.',
        'CharCNN (character-level TextCNN)', 'charcnnModel.pt', 'pytorch-charcnn',
        metrics_json(charcnn_metrics), confusion_json(charcnn_metrics), len(char_vocab),
    ),
    build_entry(
        'tinytransformer', 'Tiny Transformer (from scratch)', 'ترنسفورمر کوچک (از ابتدا)',
        'A small Transformer encoder trained from scratch (no pretraining): two self-attention '
        'layers with 4 heads over the shared word vocabulary, learned positional embeddings and '
        'a feed-forward width of 256, pooled by masked mean over real tokens, followed by dropout '
        'and a linear head. Trained from scratch on CPU on the canonical split to distinguish '
        'AI-generated content from human writing.',
        'رمزگذار کوچک ترنسفورمر آموزش‌دیده از ابتدا (بدون پیش‌آموزش): دو لایهٔ توجه خودکار با ۴ سر روی همان '
        'واژگان مشترک مدل‌های واژگانی، embeddingهای موقعیتی آموخته‌شده و بازخورد پیش‌رو با عرض ۲۵۶ که با '
        'میانگین‌گیری ماسک‌شده از توکن‌های واقعی جمع‌بندی می‌شود و پس از آن dropout و سر خطی قرار دارد. از ابتدا '
        'روی CPU و تفکیک مرجع آموزش دیده تا متن تولیدشده توسط هوش مصنوعی را از نوشته انسان جدا کند.',
        'Transformer encoder (2 layers, from scratch)', 'tinyTransformerModel.pt', 'pytorch-tinytransformer',
        metrics_json(tinytransformer_metrics), confusion_json(tinytransformer_metrics), len(tt_vocab),
    ),
]

if FORCE_RETRAIN:
    import json

    registry = json.loads(MODELS_JSON.read_text(encoding='utf-8'))
    registry = [m for m in registry if m['id'] not in {'mlp', 'charcnn', 'tinytransformer'}]
    after_laya = next(i for i, m in enumerate(registry) if m['id'] == 'laya') + 1
    registry[after_laya:after_laya] = ENTRIES
    MODELS_JSON.write_text(json.dumps(registry, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('models.json updated with:', [e['id'] for e in ENTRIES])
else:
    print('skipped (FORCE_RETRAIN = False)')


## Summary


In [ ]:
METRIC_COLS = ('accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'ece')
summary = pd.DataFrame(
    [{'split': 'v1 test', 'model': model_id, **{k: m[k] for k in METRIC_COLS}}
     for model_id, m in v1_results.items()]
    + [{'split': 'v2 cross_domain__news (sanity)', 'model': model_id, **{k: m[k] for k in METRIC_COLS}}
       for model_id, m in v2_results.items()]
)
summary
